# YOLOv8 → TFLite INT8 Cuantizado

Convierte un modelo YOLOv8 entrenado a TensorFlow Lite con cuantizacion INT8.
Optimizado para RPi Zero 2W.

Requiere: ultralytics, onnx, onnxruntime, openvino (opcional)

In [ ]:
# !pip install ultralytics onnx onnxruntime

from pathlib import Path
from ultralytics import YOLO
import numpy as np

print("Ultralytics:", YOLO.__module__)

---
## 2. Configuracion
---

In [ ]:
# Ruta al modelo YOLOv8 entrenado (.pt)
MODEL_PATH = "../Models/yolo_rgb_detector/weights/best.pt"
OUTPUT_DIR = Path("../Models_rpi")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Verificar que el modelo existe
model_path = Path(MODEL_PATH)
if not model_path.exists():
    print(f"Buscando best.pt...")
    candidates = sorted(Path("../Models").rglob("**/best.pt"))
    for c in candidates:
        print(f"  {c}")
    if candidates:
        model_path = candidates[-1]

print(f"Modelo: {model_path}")
print(f"Salida: {OUTPUT_DIR}")

---
## 3. Exportar a TFLite FP16
---

In [ ]:
model = YOLO(str(model_path))

# Exportar a TFLite (FP16 por defecto)
model.export(
    format="tflite",
    imgsz=320,
    half=True,         # FP16
    int8=False,
    data=str(Path(model_path).parent.parent / "data.yaml"),
)

# Mover al directorio de salida
tflite_fp16 = model_path.with_suffix('.tflite')
if tflite_fp16.exists():
    import shutil
    dst = OUTPUT_DIR / "yolo_detector_fp16.tflite"
    shutil.copy(str(tflite_fp16), str(dst))
    size_kb = dst.stat().st_size / 1024
    print(f"FP16 TFLite: {size_kb:.1f} KB -> {dst}")

---
## 4. Exportar a TFLite INT8
---

In [ ]:
# INT8 requiere un dataset de calibracion
model = YOLO(str(model_path))

model.export(
    format="tflite",
    imgsz=320,
    half=False,
    int8=True,
    data=str(Path(model_path).parent.parent / "data.yaml"),
)

tflite_int8 = model_path.with_suffix('.tflite')
if tflite_int8.exists():
    import shutil
    dst = OUTPUT_DIR / "yolo_detector_int8.tflite"
    shutil.copy(str(tflite_int8), str(dst))
    size_kb = dst.stat().st_size / 1024
    print(f"INT8 TFLite: {size_kb:.1f} KB -> {dst}")

---
## 5. Verificar modelos exportados
---

In [ ]:
import tensorflow as tf

for f in sorted(OUTPUT_DIR.glob("*.tflite")):
    interpreter = tf.lite.Interpreter(model_path=str(f))
    interpreter.allocate_tensors()
    
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    
    print(f"\n{f.name}:")
    print(f"  Input:  {input_details[0]['shape']} {input_details[0]['dtype']}")
    print(f"  Output: {output_details[0]['shape']} {output_details[0]['dtype']}")
    print(f"  Size:   {f.stat().st_size / 1024:.1f} KB")
    
    # Prueba de inferencia
    dummy = np.zeros(input_details[0]['shape'], dtype=np.float32)
    interpreter.set_tensor(input_details[0]['index'], dummy)
    interpreter.invoke()
    out = interpreter.get_tensor(output_details[0]['index'])
    print(f"  Inferencia de prueba: OK (output shape {out.shape})")

---
## 6. Benchmark en RPi (copiar y ejecutar en la Raspberry)
---

```python
# En la Raspberry Pi:
import tensorflow as tf
import numpy as np
import time

interpreter = tf.lite.Interpreter(model_path='yolo_detector_int8.tflite')
interpreter.allocate_tensors()

inp = interpreter.get_input_details()[0]
out = interpreter.get_output_details()[0]

img = np.random.randint(0, 255, inp['shape'], dtype=np.uint8)

times = []
for _ in range(10):
    t0 = time.time()
    interpreter.set_tensor(inp['index'], img.astype(np.float32))
    interpreter.invoke()
    result = interpreter.get_tensor(out['index'])
    times.append(time.time() - t0)

print(f'Tiempo promedio: {np.mean(times)*1000:.1f} ms')
```